# TAD-strength quartiles: WT noUV baseline vs XPC noUV (obs/exp pileups)

TADs are called on **WT noUV** insulation boundaries (`is_boundary_500000`), then
**TAD strength** = the WT noUV *domain score* (cooltools/coolpuppy rescaled on-diagonal
obs/exp pileup score, via `assign_domain_score`). TADs are split into quartiles **Q1–Q4**
by that WT-noUV strength (Q1 = weakest, Q4 = strongest).

Figure = **4×2** rescaled obs/exp pileup heatmaps (rows = strength quartiles):
- **col 1 — baseline**: $\log_2$(WT noUV obs/exp)
- **col 2 — XPC effect**: $\log_2$(XPC noUV obs/exp / WT noUV obs/exp)

Strength-quartile + rescaled-pileup machinery follows `pol2_degron_tad_q.ipynb` /
`xpc_wt_tad_q.ipynb`.

**Read col 2 as enrichment relative to each sample's own background** (per-sample obs/exp,
the field-standard cross-sample comparison) — not as absolute contact gain. Because XPC's
P(s) differs from WT, a feature's obs/exp can shift purely from the changed expected
denominator. For raw-contact differentials, use a common expected for both samples.

In [ ]:
import os
import numpy as np
import pandas as pd
import cooler
import matplotlib.pyplot as plt
from coolpuppy import coolpup

from ggner_3d.cm import CLR_CONNS, get_expected_cis_df, prepare_view_df
from ggner_3d.boundaries import make_tads, assign_domain_score
from ggner_3d import plotting
plotting.update_rcparams()

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

NPROC = 8
resolution = 10_000
window = 500_000              # insulation window used for boundary calls

# TAD length bounds
MAXLEN = 2e6
MINLEN = 2.5e5

# rescaled-pileup geometry (matches xpc_wt_tad_q / pol2_degron_tad_q)
rescale_flank = 1
rescale_size = 33 * (2 * rescale_flank + 1)
assert rescale_size % (2 * rescale_flank + 1) == 0 and rescale_size % 2 == 1, 'rescale_size'
score_flank = rescale_flank

clr_ = CLR_CONNS(resolution=resolution)
view_df = prepare_view_df(arm=False)

## 1. Call TADs on WT noUV boundaries

In [ ]:
# TADs from WT noUV insulation boundaries only
ins_wt = pd.read_csv('/home/carlos/Clone/ggner-3d/data/insulation/insulation_WTnoUV_10000bp.csv')
boundaries_wt = (
    ins_wt.loc[ins_wt[f'is_boundary_{window}'] == True, ['chrom', 'start', 'end']]
    .reset_index(drop=True)
)

tads = (
    make_tads(boundaries_wt, maxlen=MAXLEN, minlen=MINLEN)
    .sort_values(['chrom', 'start', 'end'])
    .reset_index(drop=True)
)
print(f'WT noUV boundaries: {len(boundaries_wt)}  ->  TADs: {len(tads)}')

## 2. TAD strength = WT noUV domain score → quartiles

In [ ]:
expected_ = {
    k: get_expected_cis_df(k, resolution_kb=resolution // 1000, label='chrom')
    for k in ['WTnoUV', 'XPCnoUV']
}

# strength = WT noUV domain score (rescaled on-diagonal obs/exp score)
wt_strength = assign_domain_score(
    tads,
    clr=clr_['WTnoUV'],
    expected=expected_['WTnoUV'],
    view_df=view_df,
    resolution=resolution,
    nproc=NPROC,
    clr_weight_name='sweight',
    rescale_flank=rescale_flank,
    rescale_size=rescale_size,
    score_flank=score_flank,
)

tads_q = wt_strength[['chrom', 'start', 'end', 'domain_score']].copy()
tads_q['ds_Q'] = pd.qcut(tads_q['domain_score'], q=4, labels=False) + 1  # Q1=weakest .. Q4=strongest
tads_q = tads_q.dropna(subset=['ds_Q']).reset_index(drop=True)
tads_q['ds_Q'] = tads_q['ds_Q'].astype(int)

print(tads_q['ds_Q'].value_counts().sort_index())
print(tads_q.groupby('ds_Q')['domain_score'].agg(['count', 'min', 'median', 'max']))

## 3. Per-quartile rescaled obs/exp pileups (WT noUV & XPC noUV)

In [ ]:
# pileups[q] = (WT noUV pileup, XPC noUV pileup); each ['data'][0] is the rescaled obs/exp matrix
pileups = {}
for i in range(1, 5):
    tq = tads_q[tads_q.ds_Q == i][['chrom', 'start', 'end']].reset_index(drop=True)
    cc = coolpup.CoordCreator(
        tq, resolution=resolution, features_format='bed', local=True, rescale_flank=rescale_flank
    )
    pup_wt = coolpup.PileUpper(
        clr_['WTnoUV'], cc, expected=expected_['WTnoUV'], view_df=view_df,
        ignore_diags=0, rescale_size=rescale_size, rescale=True, nproc=NPROC,
        clr_weight_name='sweight',
    ).pileupsWithControl()
    pup_xpc = coolpup.PileUpper(
        clr_['XPCnoUV'], cc, expected=expected_['XPCnoUV'], view_df=view_df,
        ignore_diags=0, rescale_size=rescale_size, rescale=True, nproc=NPROC,
        clr_weight_name='sweight',
    ).pileupsWithControl()
    pileups[i] = (pup_wt, pup_xpc)

## 4. 4×2 panel: baseline vs XPC/WT log2

In [ ]:
eps = 1e-6

# col 1 = log2(WT noUV obs/exp) ; col 2 = log2(XPC obs/exp / WT obs/exp)
base_mtx, diff_mtx = [], []
for i in range(1, 5):
    wt = pileups[i][0]['data'][0]
    xpc = pileups[i][1]['data'][0]
    base_mtx.append(np.log2(wt + eps))
    diff_mtx.append(np.log2((xpc + eps) / (wt + eps)))

# symmetric color scales (tune as needed)
base_vmin, base_vmax = -1.5, 1.5
diff_vmin, diff_vmax = -0.3, 0.3

q_sizes = tads_q['ds_Q'].value_counts().sort_index()

aspect_ratio = 5.2
fig, axs = plt.subplots(4, 2, figsize=(2 * aspect_ratio, 4 * aspect_ratio))

col_specs = [
    ('baseline\n$\\log_2$ WT noUV Obs/Exp', base_mtx, base_vmin, base_vmax, r'$\log_{2}$Obs/Exp'),
    ('XPC noUV / WT noUV\n$\\log_2$ Obs/Exp ratio', diff_mtx, diff_vmin, diff_vmax, r'$\log_{2}$FC'),
]

for col, (title, mtcs, vmin, vmax, cbar_lbl) in enumerate(col_specs):
    for i in range(1, 5):
        mtx = mtcs[i - 1]
        ax = plotting.hic_upper_triangle(
            mtx, cmap='vlag', center=0, vmin=vmin, vmax=vmax,
            ax=axs[i - 1, col], colorbar=True,
            cbar_kws=dict(fraction=0.035, pad=0.02, aspect=8),
        )
        n = mtx.shape[0]
        bins = np.array([int(n * 0.33), int(n * 0.5), int(n * 0.66)])
        ax.axis('on')
        ax.set_xticks(2 * bins, labels=["5'\nBoundary", 'Center', "3'\nBoundary"])
        ax.tick_params(axis='x', labelsize=9.5)
        ax.set_yticks([])
        ax.get_yaxis().set_visible(False)
        if i == 1:
            ax.set_title(title, fontsize=13, pad=10)
        if col == 0:
            ax.set_ylabel(f'Q{i}  n={q_sizes[i]}', fontsize=13)
            ax.get_yaxis().set_visible(True)
            ax.set_yticks([])
        cbar = ax.figure.axes[-1]
        cbar.set_ylabel(cbar_lbl, labelpad=5)
        plotting.despine(ax)

# rasterize heatmaps
for ax in axs.ravel():
    for artist in ax.get_images():
        artist.set_rasterized(True)
    for artist in ax.collections:
        artist.set_rasterized(True)

fig.tight_layout()

os.makedirs('/home/carlos/Clone/ggner-3d/figs/tads', exist_ok=True)
fig.savefig('/home/carlos/Clone/ggner-3d/figs/tads/wt_xpc_tad_q_pileups.svg', dpi=300)
fig.savefig('/home/carlos/Clone/ggner-3d/figs/tads/wt_xpc_tad_q_pileups.png', dpi=300)

## 5. TAD length distribution per strength quartile

Sanity check: WT-noUV TAD **strength anti-correlates with length** — Q1 (weakest) domains are
the *largest*, Q4 (strongest) the smallest (strong TADs are CTCF/cohesin-dense, short).
Because each quartile's pileup corner sits at a different *genomic* distance, the per-sample
obs/exp 'corner enrichment' in section 4 is partly the reciprocal of the long-range P(s)
difference — read it as enrichment-over-own-background, not raw contacts.

In [ ]:
import seaborn as sns

tads_q['length_kb'] = (tads_q['end'] - tads_q['start']) / 1e3
q_color = {1: '#465775', 2: '#5C8A6E', 3: '#F5B841', 4: '#A63446'}

print(tads_q.groupby('ds_Q')['length_kb'].agg(['count', 'mean', 'median', 'min', 'max']).round(1))

fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))

# (a) violin + box per quartile
sns.violinplot(
    data=tads_q, x='ds_Q', y='length_kb', hue='ds_Q',
    palette=q_color, inner='box', cut=0, legend=False, ax=axs[0],
)
axs[0].set_xlabel('WT noUV TAD strength quartile')
axs[0].set_ylabel('TAD length (kb)')
axs[0].set_xticks(range(4), [f'Q{i}\nn={int((tads_q.ds_Q==i).sum())}' for i in range(1, 5)])
med = tads_q.groupby('ds_Q')['length_kb'].median()
for i in range(1, 5):
    axs[0].text(i - 1, med[i] + 40, f'{med[i]:.0f}', ha='center', fontsize=9)
plotting.despine(axs[0])

# (b) cumulative distribution per quartile
for i in range(1, 5):
    v = np.sort(tads_q.loc[tads_q.ds_Q == i, 'length_kb'].values)
    axs[1].plot(v, np.linspace(0, 1, len(v)), color=q_color[i], lw=1.8, label=f'Q{i}')
axs[1].set_xlabel('TAD length (kb)')
axs[1].set_ylabel('Cumulative fraction')
axs[1].legend(title='Strength quartile', frameon=False)
plotting.despine(axs[1])

fig.tight_layout()
os.makedirs('/home/carlos/Clone/ggner-3d/figs/tads', exist_ok=True)
fig.savefig('/home/carlos/Clone/ggner-3d/figs/tads/tad_length_by_quartile.svg', dpi=300)
fig.savefig('/home/carlos/Clone/ggner-3d/figs/tads/tad_length_by_quartile.png', dpi=300)